# TOTP Multi-Factor Authentication

## Goal

This notebook demonstrates a time-based one-time password (TOTP) factor. TOTP uses a shared secret
and a time step to generate short codes.

This is still only a teaching model. Production MFA should be delivered through a mature identity
provider or MFA product, with enrollment, revocation, recovery, monitoring, and phishing-resistant
options where appropriate.

In [ ]:
import time

import pyotp


# During enrollment, the hospital server creates a per-user secret.
secret = pyotp.random_base32()
username = "dr.moretti"

# The same secret is stored by the user's authenticator app.
totp = pyotp.TOTP(secret, interval=30)

print("User:", username)
print("Enrollment secret:", secret)
print("Current code:", totp.now())

## Verifying a current code

The server accepts the code only if it matches the expected value for the current time step.

In [ ]:
# Simulate a code typed by the user from the authenticator app.
submitted_code = totp.now()

# The server verifies the submitted code against the shared secret and current time.
print("Submitted:", submitted_code)
print("Accepted?", totp.verify(submitted_code))

## Time windows and clock drift

Hospitals may allow a small adjacent time window to tolerate clock drift or network delay. A wider
window improves usability but also increases the period in which a captured code may work.

In [ ]:
current_time = int(time.time())

# Generate the previous, current, and next 30-second window codes.
for offset in [-30, 0, 30]:
    code_for_window = totp.at(current_time + offset)
    accepted_strict = totp.verify(code_for_window, for_time=current_time, valid_window=0)
    accepted_with_drift = totp.verify(code_for_window, for_time=current_time, valid_window=1)
    print(
        f"offset {offset:+4d}s",
        code_for_window,
        "strict=", accepted_strict,
        "drift_window=", accepted_with_drift,
    )

## Captured OTPs are time-limited, not phishing-proof

An OTP expires quickly, but a real-time phishing site can relay it immediately. This is why
phishing-resistant MFA, such as FIDO2/passkeys or hardware security keys, is stronger for
administrator and remote-access accounts.

In [ ]:
# Simulate a captured code.
captured_code = totp.now()
print("Captured code:", captured_code)
print("Useful right now?", totp.verify(captured_code))

# In a live notebook, wait until the code changes and run the next line again.
# The old code will eventually fail outside the accepted verification window.
print("Current code after time passes:", totp.now())

## Takeaway

TOTP is a useful second factor, but it depends on shared-secret protection, time synchronization,
safe enrollment, safe recovery, and endpoint security. It improves St. Isidore's authentication, but
it does not eliminate phishing or compromised-device risk.